Image-Text Matching and Zero-Shot (Multi-label??) Classification Using CLIP. Considering MSCOCO as the support set

# Loading the CLIP model

# 2. Prepare the Inputs

In [ ]:
# Imports and configuration
import os
import numpy as np
from PIL import Image
import requests
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

# Optional imports (heavy); wrapped in try/except so the notebook can be read/run in light environments.
try:
    import torch
    import clip
    from pycocotools.coco import COCO
except Exception as e:
    # Provide helpful message; the user should install these packages and restart the kernel.
    print("Warning: Failed to import one or more heavy libraries (torch, clip, pycocotools).")
    print("Error:", e)
    torch = None
    clip = None
    COCO = None

# Set paths (update these to your local MS-COCO locations)
IMAGE_DIR = '/path/to/val2017/'  # <-- change this
ANNOTATION_FILE = '/path/to/annotations/instances_val2017.json'  # <-- change this

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

# 80 MS-COCO class names (official)
COCO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light',
    'fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow',
    'elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee',
    'skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple',
    'sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair','couch',
    'potted plant','bed','dining table','toilet','tv','laptop','mouse','remote','keyboard','cell phone',
    'microwave','oven','toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush'
]
print(f"COCO classes loaded: {len(COCO_CLASSES)} items.")


In [ ]:
# Helper functions and evaluation (zero-shot CLIP)
import numpy as np

def safe_load_image_from_url(url):
    try:
        return Image.open(requests.get(url, stream=True).raw).convert("RGB")
    except Exception as e:
        raise RuntimeError(f"Failed to load image from {url}: {e}")

def encode_text_features(model, device, class_list):
    """Tokenize and encode class names using CLIP (if available)."""
    if clip is None or model is None:
        raise RuntimeError("CLIP model not available. Install 'git+https://github.com/openai/CLIP.git' and 'torch'.")
    text_inputs = clip.tokenize(class_list).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text_inputs)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    return text_features

def encode_image_feature(model, preprocess, image, device):
    if clip is None or model is None:
        raise RuntimeError("CLIP model not available.")
    image_input = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)
    return image_features

def predict_topk(image_features, text_features, k=5):
    # cosine similarity
    sims = (image_features @ text_features.T).cpu().numpy().squeeze()
    topk_idx = np.argsort(-sims)[:k]
    return topk_idx, sims[topk_idx]


In [ ]:
# Main evaluation driver (example)
def evaluate_zero_shot_sample(image_url, topk=5):
    if clip is None or torch is None:
        raise RuntimeError("CLIP/torch not available in this environment. This cell is for guidance; run locally.")
    # Load model if not already loaded
    global model, preprocess, text_features
    if 'model' not in globals() or model is None:
        model, preprocess = clip.load("ViT-B/32", device=DEVICE)
        text_features = encode_text_features(model, DEVICE, COCO_CLASSES)
    # Load image
    img = safe_load_image_from_url(image_url)
    image_features = encode_image_feature(model, preprocess, img, DEVICE)
    idxs, sims = predict_topk(image_features, text_features, k=topk)
    preds = [COCO_CLASSES[i] for i in idxs]
    return preds, sims

# Example usage (change the URL to an accessible COCO image URL)
if __name__ == '__main__':
    example_url = 'https://farm3.staticflickr.com/2487/3776742420_4f7b9a1c2b_z.jpg'  # sample image
    try:
        preds, sims = evaluate_zero_shot_sample(example_url, topk=5)
        print('Top predictions:', preds)
        print('Similarities:', sims)
    except Exception as e:
        print('Running example failed (expected in limited env):', e)
